# 기업 Risk Score 산출 시스템 — Flask 서버
셀을 순서대로 실행한 뒤 마지막 셀에서 서버를 시작하세요.  
브라우저에서 **http://localhost:5000** 접속

In [ ]:
from flask import Flask, render_template, jsonify, request
import pandas as pd
from pathlib import Path
import math, threading

In [ ]:
# __file__ 은 노트북에서 동작하지 않으므로 현재 작업 디렉터리 기준으로 설정
BASE          = Path.cwd()
DASHBOARD_DIR = BASE / "중간결과" / "23_대시보드"

INDUSTRIES = {
    "M03_음식료품_제조업":                  "음식료품 제조업",
    "M04_섬유_가죽_신발_제조업":             "섬유·가죽·신발 제조업",
    "M08_화학_의약품_고무_플라스틱_제조업":   "화학·의약품·고무·플라스틱 제조업",
    "M10_제1차금속산업":                    "제1차금속산업",
    "M11_조립금속제품_제조업":               "조립금속제품 제조업",
    "M12_기타기계장비_제조업":               "기타기계장비 제조업",
    "M13_전자부품_컴퓨터_전기장비_제조업":    "전자부품·컴퓨터·전기장비 제조업",
    "M15_운송장비_제조업":                  "운송장비 제조업",
    "M17_전기_가스_수도사업":               "전기·가스·수도사업",
    "M18_건설업":                          "건설업",
    "M19_도매및소매업":                    "도매 및 소매업",
    "M21_운수_창고업":                     "운수·창고업",
    "M22_정보통신업":                      "정보통신업",
    "M23_부동산_임대_사업서비스업":          "부동산·임대·사업서비스업",
    "M25_오락_문화_개인서비스업":            "오락·문화·개인서비스업",
}

SCORE_COLS = ["부실확률", "기업충격민감도", "생애주기점수", "부실확률변화", "산업충격민감도", "Porter5F"]
_cache = {}

In [ ]:
def _resolve_csv(code, suffix):
    sub  = DASHBOARD_DIR / code / f"{code}_{suffix}.csv"
    if sub.exists():  return sub
    flat = DASHBOARD_DIR / f"{code}_{suffix}.csv"
    if flat.exists(): return flat
    return None


def load_firm(code):
    if code in _cache:
        return _cache[code]
    path = _resolve_csv(code, "기업")
    if path is None:
        return pd.DataFrame()
    df = pd.read_csv(path, encoding="utf-8-sig", dtype={"사업자등록번호": str})
    df["사업자등록번호"] = df["사업자등록번호"].str.zfill(10)
    if "회사명_x" in df.columns:
        df["회사명"] = df["회사명_x"].fillna(df.get("회사명_y", ""))
    else:
        df["회사명"] = df["회사명"].fillna("")
    _cache[code] = df
    return df


def load_ind(code):
    path = _resolve_csv(code, "산업")
    if path is None:
        return pd.DataFrame()
    return pd.read_csv(path, encoding="utf-8-sig")


def safe(val):
    if val is None:
        return None
    try:
        if math.isnan(float(val)):
            return None
        return float(val)
    except Exception:
        return str(val)

In [ ]:
app = Flask(__name__, template_folder=str(BASE / "templates"))


@app.route("/")
def index():
    return render_template("dashboard.html", industries=INDUSTRIES)


@app.route("/api/companies/_all")
def api_companies_all():
    result, seen = [], set()
    for code in INDUSTRIES:
        df = load_firm(code)
        if df.empty:
            continue
        for name in df["회사명"].dropna().unique():
            if name not in seen:
                result.append({"name": name, "code": code})
                seen.add(name)
    result.sort(key=lambda x: x["name"])
    return jsonify(result)


@app.route("/api/companies/<code>")
def api_companies(code):
    df = load_firm(code)
    if df.empty:
        return jsonify([])
    return jsonify(sorted(df["회사명"].dropna().unique().tolist()))


@app.route("/api/firm/<code>")
def api_firm(code):
    company = request.args.get("company", "")
    df = load_firm(code)
    if df.empty or not company:
        return jsonify({})

    comp = df[df["회사명"] == company].sort_values("연도").reset_index(drop=True)
    if comp.empty:
        return jsonify({})

    latest = comp.iloc[-1]

    shap_by_year = {}
    for _, row in comp.iterrows():
        yr = str(int(row["연도"]))
        feats, svals = [], []
        for i in range(1, 11):
            f = row.get(f"Top{i}_Feature")
            v = row.get(f"Top{i}_SHAP_Value")
            if pd.notna(f) and pd.notna(v):
                feats.append(str(f))
                svals.append(round(float(v), 5))
        shap_by_year[yr] = {
            "features": feats, "values": svals,
            "pred_prob": safe(row.get("pred_prob")),
            "base_value": safe(row.get("base_value")),
        }

    radar_by_year = {}
    for _, row in comp.iterrows():
        yr = str(int(row["연도"]))
        radar_by_year[yr] = {c: safe(row.get(c)) or 0 for c in SCORE_COLS}

    industry_avg = {}
    avg_by_year = df.groupby("연도")[SCORE_COLS].mean(numeric_only=True)
    for yr, row_avg in avg_by_year.iterrows():
        vals = {}
        for c in SCORE_COLS:
            v = row_avg.get(c)
            vals[c] = 0.0 if pd.isna(v) else float(v)
        industry_avg[str(int(yr))] = vals

    return jsonify({
        "company":       company,
        "biz_no":        str(latest["사업자등록번호"]),
        "industry":      INDUSTRIES.get(code, code),
        "lifecycle":     str(latest["생애주기_최종"]),
        "grade":         str(latest["등급_5단계"]),
        "latest_score":  safe(latest["최종합산스코어"]),
        "latest_year":   int(latest["연도"]),
        "years":         [int(y) for y in comp["연도"]],
        "scores":        [safe(s) or 0 for s in comp["최종합산스코어"]],
        "grades":        [str(g) for g in comp["등급_5단계"]],
        "lifecycles":    [str(l) for l in comp["생애주기_최종"]],
        "radar_by_year": radar_by_year,
        "industry_avg":  industry_avg,
        "shap":          shap_by_year,
    })


@app.route("/api/industry/<code>")
def api_industry(code):
    df = load_ind(code)
    if df.empty:
        return jsonify([])
    return jsonify(df[["Combined_Rank", "Feature", "Category"]].to_dict("records"))


@app.route("/api/grade_dist/<code>")
def api_grade_dist(code):
    df = load_firm(code)
    if df.empty:
        return jsonify({})
    yr  = int(df["연도"].max())
    sub = df[df["연도"] == yr]
    order  = ["매우 위험", "위험", "중립", "안정", "매우 안정"]
    counts = sub["등급_5단계"].value_counts().reindex(order, fill_value=0).to_dict()
    top = (
        sub[sub["등급_5단계"].isin(["매우 위험", "위험"])]
        [["회사명", "최종합산스코어", "등급_5단계", "생애주기_최종"]]
        .sort_values("최종합산스코어", ascending=False)
        .head(10).to_dict("records")
    )
    return jsonify({"year": yr, "counts": counts, "top_risk": top})

In [ ]:
# 서버 시작 (백그라운드 스레드로 실행해 노트북이 블로킹되지 않음)
# 이미 실행 중이면 이 셀을 다시 실행하지 마세요
t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True)
t.start()
print("서버 시작: http://localhost:5000")